# Generate DFDCP Corruption Embeddings

Chỉ cần đổi `LEVEL`. Notebook sẽ tự tìm dataset `/kaggle/input/dfdcp-corruption-level-{LEVEL}`, dò 4 corruption folder, trích CLIP `ViT-L-14/openai`, rồi lưu file `.pt` cho từng corruption.

In [ ]:
%pip install -q open_clip_torch

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import open_clip

# ===== Chỉ cần đổi tham số này =====
LEVEL = 4

# Các tham số dưới đây giữ giống pipeline hiện tại trong repo.
CORRUPTIONS = ["color_contrast", "color_saturation", "gaussian_blur", "resize"]
CLIP_MODEL = "ViT-L-14"
PRETRAINED = "openai"
BATCH_SIZE = 128
NUM_WORKERS = 2
USE_AMP = True
SKIP_EXISTING = True

INPUT_ROOT = Path(f"/kaggle/input/dfdcp-corruption-level-{LEVEL}")
OUTPUT_DIR = Path(f"/kaggle/working/dfdcp_level{LEVEL}_clip_embeddings")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("LEVEL:", LEVEL)
print("input root:", INPUT_ROOT)
print("output dir:", OUTPUT_DIR)
print("device:", device)

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def resolve_input_root(level: int) -> Path:
    preferred = Path(f"/kaggle/input/dfdcp-corruption-level-{level}")
    if preferred.exists():
        return preferred

    matches = sorted(Path("/kaggle/input").glob(f"*dfdcp*level*{level}*"))
    if matches:
        return matches[0]

    raise FileNotFoundError(
        f"Không tìm thấy Kaggle input cho level {level}. "
        f"Hãy add dataset dfdcp-corruption-level-{level} vào notebook."
    )


def has_images(folder: Path) -> bool:
    return any(p.is_file() and p.suffix.lower() in IMAGE_EXTS for p in folder.rglob("*"))


def find_corruption_dir(root: Path, corruption: str) -> Path:
    candidates = [root / corruption, root / f"dfdcp-corruption-level-{LEVEL}" / corruption]

    # Levels 1-3 trên Kaggle thường có dạng _output_*/processed_output/<corruption>.
    patterns = [
        f"*/processed_output/{corruption}",
        f"*/*/processed_output/{corruption}",
        f"processed_output/{corruption}",
    ]
    for pattern in patterns:
        candidates.extend(p for p in root.glob(pattern) if p.is_dir())

    if not any(candidate.exists() for candidate in candidates):
        candidates.extend(p for p in root.rglob(corruption) if p.is_dir())

    seen = set()
    valid = []
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if candidate.exists() and has_images(candidate):
            valid.append(candidate)

    if not valid:
        raise FileNotFoundError(f"Không tìm thấy folder có ảnh cho corruption: {corruption}")
    if len(valid) > 1:
        print(f"Có nhiều candidate cho {corruption}, dùng candidate đầu tiên:")
        for path in valid:
            print(" -", path)
    return valid[0]


def list_images(folder: Path) -> list[Path]:
    return sorted(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def infer_label(path: Path) -> int:
    parts = [part.lower() for part in path.parts]
    fake_parts = {"fake", "fakes", "manipulated", "manipulated_sequences", "forged", "tampered"}
    real_parts = {"real", "reals", "original", "original_sequences", "pristine", "authentic"}

    if any(part in fake_parts or part.startswith("fake") for part in parts):
        return 1
    if any(part in real_parts or part.startswith("real") for part in parts):
        return 0

    text = "/".join(parts)
    if "manipulated" in text or "/fake" in text:
        return 1
    if "original" in text or "/real" in text:
        return 0

    raise ValueError(f"Không suy ra được label từ path: {path}")


class ImagePathDataset(Dataset):
    def __init__(self, paths: list[Path], preprocess):
        self.paths = [Path(p) for p in paths]
        self.preprocess = preprocess

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        image = Image.open(path).convert("RGB")
        label = infer_label(path)
        return self.preprocess(image), label, str(path)


@torch.inference_mode()
def extract_features(paths: list[Path], model, preprocess):
    loader = DataLoader(
        ImagePathDataset(paths, preprocess),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device == "cuda"),
        persistent_workers=(NUM_WORKERS > 0),
    )

    features, labels, saved_paths = [], [], []
    for images, batch_labels, batch_paths in tqdm(loader, total=len(loader)):
        images = images.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(USE_AMP and device == "cuda")):
            batch_features = model.encode_image(images)
        features.append(F.normalize(batch_features.float(), dim=-1).cpu())
        labels.append(batch_labels.cpu())
        saved_paths.extend(batch_paths)

    return torch.cat(features), torch.cat(labels), saved_paths


def save_feature_file(output_path: Path, features: torch.Tensor, labels: torch.Tensor, paths: list[str], metadata: dict):
    payload = {
        "features": features,
        "labels": labels,
        "paths": paths,
        **metadata,
    }
    torch.save(payload, output_path)
    print("saved:", output_path)
    print("features:", tuple(features.shape))
    print("labels:", tuple(labels.shape))
    print("counts [REAL, FAKE]:", torch.bincount(labels.long(), minlength=2).tolist())

In [ ]:
INPUT_ROOT = resolve_input_root(LEVEL)
corruption_dirs = {name: find_corruption_dir(INPUT_ROOT, name) for name in CORRUPTIONS}

print("Resolved corruption folders:")
for name, folder in corruption_dirs.items():
    image_count = len(list_images(folder))
    print(f"- {name}: {folder} | images={image_count:,}")

In [ ]:
print("creating CLIP:", f"{CLIP_MODEL}/{PRETRAINED}")
model, _, preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL,
    pretrained=PRETRAINED,
    device=device,
)
model.eval()
for param in model.parameters():
    param.requires_grad_(False)

if device == "cuda":
    torch.backends.cudnn.benchmark = True
    print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
print("CLIP ready")

In [ ]:
saved_files = []

for corruption, folder in corruption_dirs.items():
    output_path = OUTPUT_DIR / f"dfdcp_level{LEVEL}_{corruption}_features.pt"
    if SKIP_EXISTING and output_path.exists():
        print("skip existing:", output_path)
        saved_files.append(output_path)
        continue

    paths = list_images(folder)
    print(f"\nExtracting DFDCP | level {LEVEL} | {corruption} | images={len(paths):,}")

    # Check label parsing before the long CLIP pass.
    sample_labels = [infer_label(path) for path in paths[: min(len(paths), 2000)]]
    print("sample counts [REAL, FAKE]:", torch.bincount(torch.tensor(sample_labels), minlength=2).tolist())

    features, labels, saved_paths = extract_features(paths, model, preprocess)
    save_feature_file(
        output_path,
        features,
        labels,
        saved_paths,
        metadata={
            "dataset_name": "DFDCP",
            "split_name": "test",
            "transform_name": corruption,
            "transform_level": LEVEL,
            "clip_model": f"{CLIP_MODEL}/{PRETRAINED}",
            "input_root": str(INPUT_ROOT),
            "corruption_root": str(folder),
        },
    )
    saved_files.append(output_path)

print("\nSaved files:")
for path in saved_files:
    print("-", path)

In [ ]:
zip_path = Path("/kaggle/working") / f"dfdcp_level{LEVEL}_clip_embeddings.zip"
with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as zf:
    for path in saved_files:
        zf.write(path, arcname=path.name)

print("zip:", zip_path)
print("done")